# Chile Reaches With Valid SWOT DAWG Discharge

This notebook:

1. Finds an existing South America DAWG SOS netCDF or downloads it with `earthaccess`.
2. Reads Chile `reach_id` values from `chile_reaches.gpkg`.
3. Checks those reaches against the SOS `consensus_q` variable.
4. Saves a CSV listing the Chile reaches that have at least one valid discharge measurement.


In [ ]:
from pathlib import Path
import json

import earthaccess
import geopandas as gpd
import netCDF4 as nc
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)


In [ ]:
# Inputs
SWORD_GPKG = Path(r"C:\SWOT_universal_PIXC_quantile_filter\data\Petrohue_SWORD_reaches\chile_reaches.gpkg")
DOWNLOAD_DIR = Path(r"C:\UNESCO\Code\downloaded_files")
PREFERRED_NC_PATH = None  # set this to a specific SA netCDF path if you want to bypass search/download
OUT_CSV = Path(r"C:\SWOT_universal_PIXC_quantile_filter\data\Petrohue_SWORD_reaches\chile_reaches_with_valid_discharge.csv")

SHORT_NAME = "SWOT_L4_HR_DAWG_SOS_DISCHARGE_V3"
NATIVE_ID_PREFIX = "sa_sword_v16_"
SWORD_REACH_COL = "reach_id"
ALG_GROUP = "consensus"
Q_VAR_NAME = "consensus_q"
TIME_VAR_NAME = "time_int"
SENTINEL = -9.0e10
MIN_VALID_POINTS = 1

SWORD_GPKG, DOWNLOAD_DIR, OUT_CSV


In [ ]:
def locate_or_download_sa_sos(preferred_path=None, download_dir=DOWNLOAD_DIR):
    if preferred_path is not None:
        preferred_path = Path(preferred_path)
        if preferred_path.exists():
            return preferred_path
        raise FileNotFoundError(f"Preferred netCDF does not exist: {preferred_path}")

    existing = sorted(download_dir.rglob(f"{NATIVE_ID_PREFIX}*.nc"), key=lambda p: p.stat().st_mtime, reverse=True)
    if existing:
        print(f"Using existing SA netCDF: {existing[0]}")
        return existing[0]

    print("No local South America DAWG SOS file found. Attempting download with earthaccess...")
    earthaccess.login(strategy="netrc")
    granules = earthaccess.search_data(
        short_name=SHORT_NAME,
        sort_key="-start_date",
        count=100,
    )

    sa_granules = [g for g in granules if g["meta"]["native-id"].startswith(NATIVE_ID_PREFIX)]
    if not sa_granules:
        raise FileNotFoundError(f"No granule starting with {NATIVE_ID_PREFIX!r} was returned by earthaccess.")

    paths = earthaccess.download([sa_granules[0]], local_path=str(download_dir))
    nc_path = Path(paths[0])
    print(f"Downloaded SA netCDF: {nc_path}")
    return nc_path


def decode_text(value):
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="ignore")
    return str(value)


def scalar_missing_value(var_obj):
    missing = None
    if "_FillValue" in var_obj.ncattrs():
        missing = var_obj.getncattr("_FillValue")
    elif "missing_value" in var_obj.ncattrs():
        missing = var_obj.getncattr("missing_value")

    if missing is not None and np.ndim(missing) > 0:
        missing = np.array(missing).ravel()[0]
    return missing


In [ ]:
# 1) Load Chile reach IDs from the GeoPackage
gdf = gpd.read_file(SWORD_GPKG)
gdf[SWORD_REACH_COL] = pd.to_numeric(gdf[SWORD_REACH_COL], errors="coerce")
gdf = gdf.dropna(subset=[SWORD_REACH_COL]).copy()
gdf[SWORD_REACH_COL] = gdf[SWORD_REACH_COL].astype("int64")

chile_reach_ids = set(gdf[SWORD_REACH_COL].unique())
print(f"Chile reaches in GeoPackage: {len(chile_reach_ids):,}")
gdf[[SWORD_REACH_COL]].drop_duplicates().head()


In [ ]:
# 2) Locate or download the South America SOS file
NC_PATH = locate_or_download_sa_sos(PREFERRED_NC_PATH, DOWNLOAD_DIR)
NC_PATH


In [ ]:
# 3) Check Chile reaches against consensus discharge
records = []
n_chile_reaches_seen_in_nc = 0

with nc.Dataset(NC_PATH, "r") as ds:
    reaches = ds.groups["reaches"]
    consensus = ds.groups[ALG_GROUP]

    reach_ids_nc = np.asarray(reaches.variables["reach_id"][:], dtype="int64")
    river_names_nc = reaches.variables["river_name"][:]

    t_var = consensus.variables[TIME_VAR_NAME]   # variable-length array per reach
    q_var = consensus.variables[Q_VAR_NAME]      # variable-length array per reach
    missing = scalar_missing_value(q_var)

    for i, rid in enumerate(reach_ids_nc):
        rid = int(rid)
        if rid not in chile_reach_ids:
            continue

        n_chile_reaches_seen_in_nc += 1

        times = np.asarray(t_var[i], dtype="float64")
        q = np.asarray(q_var[i], dtype="float64")

        valid = np.isfinite(times) & (times > SENTINEL) & np.isfinite(q) & (q > SENTINEL)
        if missing is not None:
            valid = valid & (q != missing)

        n_valid = int(np.sum(valid))
        if n_valid < MIN_VALID_POINTS:
            continue

        records.append(
            {
                "reach_id": rid,
                "river_name": decode_text(river_names_nc[i]),
                "n_valid_obs": n_valid,
                "first_valid_datetime_utc": pd.Timestamp("2000-01-01") + pd.to_timedelta(int(times[valid][0]), unit="s"),
                "last_valid_datetime_utc": pd.Timestamp("2000-01-01") + pd.to_timedelta(int(times[valid][-1]), unit="s"),
            }
        )

df = pd.DataFrame.from_records(records)
if not df.empty:
    df = df.sort_values(["n_valid_obs", "reach_id"], ascending=[False, True]).reset_index(drop=True)

print(f"Chile reaches found in SA netCDF: {n_chile_reaches_seen_in_nc:,}")
print(f"Chile reaches with >= {MIN_VALID_POINTS} valid consensus observations: {len(df):,}")
df.head(10)


In [ ]:
# 4) Save CSV of reach IDs with valid discharge measurements
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}")
print(f"Rows: {len(df):,}")
OUT_CSV
